# 🚀 CE-EEN-B0: High-Performance Skin Disease Classification

## 🎯 Optimization Goals
- **Maximize Accuracy**: Unfreezing EfficientNetB0 for fine-tuning
- **Maximize GPU Usage**: Increased batch size & mixed precision
- **Dataset**: Massive Skin Disease Balanced Dataset (262k images, 34 classes)

## ⚙️ Key Config Changes
- **Batch Size**: 64 (High GPU Load)
- **Fine-Tuning**: EfficientNetB0 layers unfrozen
- **LR Schedule**: Cosine Decay with Warmup
- **Label Smoothing**: 0.1 (Better Generalization)
- **Loss Function**: Categorical Crossentropy (One-Hot Encoded)
- **Visualization**: Live training plots enabled

---

In [ ]:
# Install visualization tool
!pip install -q livelossplot

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from livelossplot import PlotLossesKeras

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, balanced_accuracy_score

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")

In [ ]:
# ==========================================
# 🚀 STRICT GPU CONFIGURATION
# ==========================================
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError("❌ NO GPU DETECTED! Stopping execution. Please enable GPU in Settings.")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

mixed_precision.set_global_policy('mixed_float16')
print(f"✓ GPU Active: {len(gpus)} devices (Training will happen on GPU)")
print("✓ Mixed Precision: ON (Float16 Tensor Cores)")

# Config
SEED = 42
IMG_SIZE = 180
BATCH_SIZE = 64      # Increased for 100% GPU Usage
EPOCHS = 40          # Increased for better convergence
LR_START = 1e-5      # Warmup start
LR_MAX = 1e-3        # Peak LR
LABEL_SMOOTH = 0.1   # Helps generalization

# Dataset path - adjust based on your Kaggle dataset
BASE_DIR = Path('/kaggle/input/massive-skin-disease-balanced-dataset')
MODEL_DIR = Path('/kaggle/working/models')
MODEL_DIR.mkdir(exist_ok=True, parents=True)

tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"\n📁 Base Directory: {BASE_DIR}")
print(f"📁 Model Output Directory: {MODEL_DIR}")

## 🔍 Dataset Structure Explorer
Let's explore the dataset structure to understand how it's organized.

In [ ]:
# ==========================================
# EXPLORE DATASET STRUCTURE
# ==========================================

def explore_directory(path, max_depth=3, current_depth=0, max_items=10):
    """Recursively explore directory structure"""
    if current_depth >= max_depth:
        return
    
    indent = "  " * current_depth
    items = list(path.iterdir())[:max_items]
    
    for item in items:
        if item.is_dir():
            # Count files in directory
            try:
                file_count = len(list(item.glob('*.*')))
                print(f"{indent}📁 {item.name}/ ({file_count} files)")
                if current_depth < 2:  # Only go deeper for first 2 levels
                    explore_directory(item, max_depth, current_depth + 1, max_items=5)
            except:
                print(f"{indent}📁 {item.name}/")
        else:
            print(f"{indent}📄 {item.name}")
    
    if len(list(path.iterdir())) > max_items:
        print(f"{indent}... and {len(list(path.iterdir())) - max_items} more items")

print("Dataset Structure:")
print("="*60)
explore_directory(BASE_DIR)
print("="*60)

## 1️⃣ Data Loading & Preprocessing

In [ ]:
# ==========================================
# SMART DATASET DETECTION WITH DEEP SEARCH
# ==========================================

def find_image_folders(root_path, max_depth=3):
    """Recursively search for folders containing images"""
    image_extensions = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}
    
    def search_recursive(path, depth=0):
        if depth > max_depth:
            return None
        
        # Get all subdirectories
        subdirs = [d for d in path.iterdir() if d.is_dir()]
        
        if not subdirs:
            return None
        
        # Check if subdirectories contain images
        folders_with_images = []
        for subdir in subdirs:
            # Check if this folder has images
            files = list(subdir.iterdir())[:100]  # Sample first 100 files
            has_images = any(f.suffix in image_extensions for f in files if f.is_file())
            
            if has_images:
                folders_with_images.append(subdir)
        
        # If we found folders with images, return the parent
        if len(folders_with_images) >= 3:  # At least 3 class folders
            return path
        
        # Otherwise, search deeper
        for subdir in subdirs:
            result = search_recursive(subdir, depth + 1)
            if result:
                return result
        
        return None
    
    return search_recursive(root_path)

# Check if base directory exists
if not BASE_DIR.exists():
    raise FileNotFoundError(
        f"❌ Dataset directory not found: {BASE_DIR}\n"
        "Please ensure the dataset is added to your Kaggle notebook"
    )

print("🔍 Searching for image folders...")
DATA_DIR = find_image_folders(BASE_DIR)

if DATA_DIR is None:
    raise ValueError(
        "❌ Could not locate dataset with class folders containing images!\n"
        f"Searched in: {BASE_DIR}\n"
        "Expected structure: /path/to/dataset/class_name/image.jpg\n\n"
        "Please check the dataset structure above and verify images are organized in class folders."
    )

print(f"\n✓ Found dataset at: {DATA_DIR}")
print(f"  Relative path: {DATA_DIR.relative_to(BASE_DIR)}")

In [ ]:
# ==========================================
# LOAD CLASS FOLDERS AND IMAGES
# ==========================================

# Get class list
class_folders = sorted([f for f in DATA_DIR.iterdir() if f.is_dir()])
if len(class_folders) == 0:
    raise ValueError(
        f"❌ No class folders found in {DATA_DIR}\n"
        f"Directory contents: {list(DATA_DIR.iterdir())[:10]}"
    )

class_names = [f.name for f in class_folders]
num_classes = len(class_names)
print(f"✓ Found {num_classes} classes")
print(f"\nFirst 10 classes:")
for i, name in enumerate(class_names[:10], 1):
    print(f"  {i}. {name}")
if num_classes > 10:
    print(f"  ... and {num_classes - 10} more")

# Collect all file paths
print(f"\nLoading images from {num_classes} classes...")
paths, labels = [], []
image_extensions = ['.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG']

for idx, folder in enumerate(class_folders):
    # Support multiple image formats
    imgs = []
    for ext in image_extensions:
        imgs.extend(list(folder.glob(f'*{ext}')))
    
    if len(imgs) > 0:
        paths.extend([str(p) for p in imgs])
        labels.extend([idx] * len(imgs))
        if idx < 5:  # Show first 5 classes
            print(f"  {folder.name}: {len(imgs):,} images")

if len(class_folders) > 5:
    print(f"  ... loading from {len(class_folders) - 5} more classes")

paths = np.array(paths)
labels = np.array(labels)

# Final verification
if len(paths) == 0:
    raise ValueError(
        "❌ No images found in dataset!\n"
        "Please check:\n"
        "1. Dataset contains .jpg or .png files\n"
        "2. Images are inside class folders\n"
        "3. Dataset structure: /dataset/class_name/image.jpg"
    )

print(f"\n{'='*60}")
print(f"✓ Total Images: {len(paths):,}")
print(f"✓ Total Classes: {num_classes}")
print(f"✓ Images per class (avg): {len(paths)//num_classes:,}")
print(f"{'='*60}")

In [ ]:
# ==========================================
# TRAIN/VAL/TEST SPLIT (70/15/15)
# ==========================================

# Split Data
train_p, temp_p, train_l, temp_l = train_test_split(
    paths, labels, test_size=0.3, stratify=labels, random_state=SEED
)
val_p, test_p, val_l, test_l = train_test_split(
    temp_p, temp_l, test_size=0.5, stratify=temp_l, random_state=SEED
)

print(f"✓ Train: {len(train_p):,} images ({len(train_p)/len(paths)*100:.1f}%)")
print(f"✓ Val:   {len(val_p):,} images ({len(val_p)/len(paths)*100:.1f}%)")
print(f"✓ Test:  {len(test_p):,} images ({len(test_p)/len(paths)*100:.1f}%)")

In [ ]:
# ==========================================
# DATA PIPELINE WITH PREPROCESSING
# ==========================================

def extract_contour_and_crop(path_bytes):
    """Extract main contour and crop image for better focus"""
    try:
        # Decode bytes to string
        path = path_bytes.numpy().decode('utf-8')
        img = cv2.imread(path)
        
        if img is None:
            return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if not contours:
            return cv2.resize(rgb, (IMG_SIZE, IMG_SIZE))
        
        cnt = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(cnt)
        
        # Add margin
        margin = int(0.05 * max(w, h))
        x, y = max(0, x-margin), max(0, y-margin)
        w = min(img.shape[1]-x, w+2*margin)
        h = min(img.shape[0]-y, h+2*margin)
        
        cropped = rgb[y:y+h, x:x+w]
        return cv2.resize(cropped, (IMG_SIZE, IMG_SIZE))
    
    except Exception as e:
        return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

def preprocess(path, label):
    """Preprocess image and convert label to one-hot"""
    img = tf.py_function(extract_contour_and_crop, [path], tf.uint8)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    img = tf.cast(img, tf.float32) / 255.0
    
    # ONE-HOT ENCODING for Label Smoothing
    label = tf.one_hot(label, num_classes)
    return img, label

def augment(img, label):
    """Apply data augmentation"""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, tf.random.uniform([], 0, 4, tf.int32))
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    return tf.clip_by_value(img, 0, 1), label

def make_ds(paths, labels, aug=False):
    """Create optimized tf.data.Dataset"""
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if aug:
        ds = ds.shuffle(20000, seed=SEED)
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if aug:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Create datasets
print("Creating data pipelines...")
train_ds = make_ds(train_p, train_l, aug=True)
val_ds = make_ds(val_p, val_l)
test_ds = make_ds(test_p, test_l)

print("\n✓ Data Pipeline Optimized (One-Hot Encoded)")
print(f"✓ Training batches: {len(train_ds)}")
print(f"✓ Validation batches: {len(val_ds)}")
print(f"✓ Test batches: {len(test_ds)}")

## 2️⃣ Advanced Model Architecture (Unfrozen)

In [ ]:
def build_model(n_classes):
    """Build EfficientNetB0 model with custom head"""
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    
    # EfficientNetB0 - Unfrozen for Fine-Tuning
    base = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=inp)
    base.trainable = True  # 🔓 UNFREEZE for maximum accuracy
    
    x = base.output
    
    # Custom Head
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    
    # Output (Float32 for numerical stability)
    out = layers.Dense(n_classes, activation='softmax', dtype='float32')(x)
    
    return models.Model(inp, out, name='CE_EEN_B0_FineTuned')

model = build_model(num_classes)
model.summary()

## 3️⃣ Cosine Decay LR Schedule

In [ ]:
# Cosine Decay with Warmup
def lr_schedule(epoch):
    """Learning rate schedule with warmup and cosine decay"""
    warmup_epochs = 3
    if epoch < warmup_epochs:
        return LR_START + (LR_MAX - LR_START) * (epoch / warmup_epochs)
    else:
        progress = (epoch - warmup_epochs) / (EPOCHS - warmup_epochs)
        return LR_MAX * 0.5 * (1 + np.cos(np.pi * progress))

lr_callback = keras.callbacks.LearningRateScheduler(lr_schedule)

# Plot LR Schedule
plt.figure(figsize=(10, 4))
plt.plot([lr_schedule(e) for e in range(EPOCHS)], linewidth=2)
plt.title('Learning Rate Schedule', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.grid(True, alpha=0.3)
plt.show()

## 4️⃣ Train with Live Visualization

In [ ]:
# Compile model
model.compile(
    optimizer=keras.optimizers.Adamax(learning_rate=LR_START), 
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
    metrics=['accuracy']
)

# Using .keras format to avoid mixed precision serialization issues
checkpoint_path = str(MODEL_DIR / 'best_model.keras')

callbacks = [
    ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    lr_callback,
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    PlotLossesKeras(),  # 📈 Live Plotting
    keras.callbacks.CSVLogger(str(MODEL_DIR/'training_log.csv'))  # 💾 Log to CSV
]

print(f"\n{'='*60}")
print(f"Training Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Initial LR: {LR_START}")
print(f"  Max LR: {LR_MAX}")
print(f"  Label Smoothing: {LABEL_SMOOTH}")
print(f"  Mixed Precision: ON")
print(f"  Classes: {num_classes}")
print(f"{'='*60}\n")

print("🚀 Starting training...")
print("Live plots will appear below:\n")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Training Complete!")

## 5️⃣ Evaluation & Saving

In [ ]:
# Load best model (.keras format)
print("Loading best model...")
model = keras.models.load_model(str(MODEL_DIR/'best_model.keras'))

# Evaluate on test set
print("\nEvaluating on test set...")
loss, acc = model.evaluate(test_ds, verbose=1)
print(f"\n{'='*60}")
print(f"🏆 Test Loss: {loss:.4f}")
print(f"🏆 Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print(f"{'='*60}")

In [ ]:
# Detailed Classification Report
print("\nGenerating classification report...")
y_true, y_pred = [], []

for imgs, labs in test_ds:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labs.numpy(), axis=1))  # Decode one-hot
    y_pred.extend(np.argmax(preds, axis=1))

print("\n" + "="*80)
print("CLASSIFICATION REPORT")
print("="*80)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

# Balanced Accuracy
bal_acc = balanced_accuracy_score(y_true, y_pred)
print(f"\nBalanced Accuracy: {bal_acc:.4f} ({bal_acc*100:.2f}%)")

In [ ]:
# Save Final Model and Classes
print("\nSaving final model...")
model.save(str(MODEL_DIR/'final_model.keras'))
np.save(str(MODEL_DIR/'classes.npy'), class_names)

print("\n✓ All models saved in .keras format")
print(f"  - Best model: {MODEL_DIR/'best_model.keras'}")
print(f"  - Final model: {MODEL_DIR/'final_model.keras'}")
print(f"  - Classes: {MODEL_DIR/'classes.npy'}")
print(f"  - Training log: {MODEL_DIR/'training_log.csv'}")

## 6️⃣ 📸 Inference Demo (Path Input)
Provide a path to an image to classify it.

In [ ]:
# Load Model & Classes
loaded_model = keras.models.load_model(str(MODEL_DIR/'best_model.keras'))
loaded_classes = np.load(str(MODEL_DIR/'classes.npy'))

def predict_image_path(image_path):
    """Predict skin disease from image path"""
    if not os.path.exists(image_path):
        print(f"❌ Error: File not found at {image_path}")
        return
        
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        print("❌ Error: Could not read image file")
        return
        
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Preprocess (Contour + Resize)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(cnt)
        margin = int(0.05 * max(w, h))
        x, y = max(0, x-margin), max(0, y-margin)
        w = min(img.shape[1]-x, w+2*margin)
        h = min(img.shape[0]-y, h+2*margin)
        img_crop = img_rgb[y:y+h, x:x+w]
    else:
        img_crop = img_rgb
        
    img_resized = cv2.resize(img_crop, (IMG_SIZE, IMG_SIZE))
    img_batch = np.expand_dims(img_resized / 255.0, axis=0)
    
    # Predict
    preds = loaded_model.predict(img_batch, verbose=0)
    idx = np.argmax(preds)
    class_name = loaded_classes[idx]
    conf = preds[0][idx]
    
    # Display
    plt.figure(figsize=(8, 8))
    plt.imshow(img_crop)
    plt.title(f"Prediction: {class_name}\nConfidence: {conf:.2%}", 
              fontsize=16, color='green' if conf > 0.8 else 'orange', fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Print top 3 predictions
    top3_idx = np.argsort(preds[0])[-3:][::-1]
    print("\nTop 3 Predictions:")
    for i, idx in enumerate(top3_idx, 1):
        print(f"  {i}. {loaded_classes[idx]}: {preds[0][idx]:.2%}")

# Example Usage
print("👇 Call the function with an image path:")
print("predict_image_path('path/to/your/image.jpg')")
print("\nExample:")
print(f"predict_image_path('{str(DATA_DIR)}/[class_name]/image.jpg')")